In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip uninstall -y pyspark dataproc-spark-connect

In [ ]:
!pip uninstall -y dataproc-spark-connect pyspark
!pip install -q pyspark==3.5.1

In [ ]:
import os
import pyspark
print("PySpark version:", pyspark.__version__)
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CCF-Colab")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

print(spark.version)

In [ ]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel
import time

In [ ]:
data = [
    ("A", "B"),
    ("B", "C"),
    ("B", "D"),
    ("D", "E"),
    ("F", "G"),
    ("G", "H")
]

df_edges = spark.createDataFrame(data, ["src", "dst"])
df_edges.show(truncate=False)

In [ ]:
def make_undirected(df_edges: DataFrame) -> DataFrame:
    cleaned = (
        df_edges
        .filter(F.col("src").isNotNull() & F.col("dst").isNotNull())
        .filter(F.col("src") != F.col("dst"))
        .dropDuplicates()
    )

    reversed_df = cleaned.select(
        F.col("dst").alias("src"),
        F.col("src").alias("dst")
    )

    return cleaned.union(reversed_df).dropDuplicates()

In [ ]:
df_undirected = make_undirected(df_edges).cache()
df_undirected.show(truncate=False)

In [ ]:
nodes_df = (
    df_undirected.select(F.col("src").alias("node"))
    .union(df_undirected.select(F.col("dst").alias("node")))
    .dropDuplicates()
    .cache()
)

nodes_df.show(truncate=False)

In [ ]:
undirected_rdd = df_undirected.rdd.map(lambda r: (r["src"], r["dst"])).cache()
print(sorted(undirected_rdd.collect()))

In [ ]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel

def run_ccf_rdd_fast(
    df_undirected: DataFrame,
    max_iter: int = 100,
    verbose: bool = False,
    num_partitions: int | None = None
):
    sc = spark.sparkContext
    if num_partitions is None:
        num_partitions = sc.defaultParallelism

    # Base graph
    undirected_rdd = (
        df_undirected
        .select("src", "dst")
        .rdd
        .map(lambda r: (r["src"], r["dst"]))
        .partitionBy(num_partitions)
        .persist(StorageLevel.MEMORY_AND_DISK)
    )
    undirected_rdd.count()

    # Re-key by destination to propagate labels from neighbor -> source
    graph_by_dst_rdd = (
        undirected_rdd
        .map(lambda x: (x[1], x[0]))
        .partitionBy(num_partitions)
        .persist(StorageLevel.MEMORY_AND_DISK)
    )
    graph_by_dst_rdd.count()

    # Initial labels: node -> node
    labels_rdd = (
        undirected_rdd
        .flatMap(lambda x: [x[0], x[1]])
        .distinct(num_partitions)
        .map(lambda node: (node, node))
        .partitionBy(num_partitions)
        .persist(StorageLevel.MEMORY_AND_DISK)
    )
    labels_rdd.count()

    changed = True
    iteration = 0

    while changed and iteration < max_iter:
        iteration += 1

        propagated = (
            graph_by_dst_rdd
            .join(labels_rdd)                     # (dst, (src, label_of_dst))
            .map(lambda x: (x[1][0], x[1][1]))   # (src, neighbor_label)
            .partitionBy(num_partitions)
        )

        min_neighbor_labels = (
            propagated
            .reduceByKey(lambda a, b: a if a < b else b, numPartitions=num_partitions)
            .persist(StorageLevel.MEMORY_AND_DISK)
        )

        next_labels_rdd = (
            labels_rdd
            .leftOuterJoin(min_neighbor_labels)
            .mapValues(lambda x: x[1] if x[1] is not None and x[1] < x[0] else x[0])
            .partitionBy(num_partitions)
            .persist(StorageLevel.MEMORY_AND_DISK)
        )

        # Single action to materialize next_labels_rdd
        next_labels_rdd.count()

        num_changes = (
            labels_rdd
            .join(next_labels_rdd)
            .filter(lambda x: x[1][0] != x[1][1])
            .count()
        )

        if verbose:
            print(f"Iteration {iteration} -> changed labels: {num_changes}")

        changed = num_changes > 0

        labels_rdd.unpersist()
        min_neighbor_labels.unpersist()
        labels_rdd = next_labels_rdd

    return labels_rdd, iteration

In [ ]:
final_labels_rdd, rdd_iterations = run_ccf_rdd_fast(df_undirected, max_iter=20, verbose=True)

print("RDD iterations =", rdd_iterations)
print(sorted(final_labels_rdd.collect(), key=lambda x: x[0]))

In [ ]:
spark.sparkContext.setCheckpointDir("/content/spark-checkpoints")

In [ ]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel

def run_ccf_dataframe_fast(
    df_undirected: DataFrame,
    max_iter: int = 100,
    verbose: bool = False,
    num_partitions: int | None = None,
    checkpoint_every: int = 5,
    broadcast_threshold: int = 50000
):
    sc = spark.sparkContext
    if num_partitions is None:
        num_partitions = sc.defaultParallelism

    # Repartition graph once by dst, because we join on dst == node
    df_graph = (
        df_undirected
        .select("src", "dst")
        .repartition(num_partitions, "dst")
        .persist(StorageLevel.MEMORY_AND_DISK)
    )
    df_graph.count()

    nodes_df = (
        df_graph.select(F.col("src").alias("node"))
        .union(df_graph.select(F.col("dst").alias("node")))
        .dropDuplicates(["node"])
        .repartition(num_partitions, "node")
    )

    labels_df = (
        nodes_df
        .withColumn("label", F.col("node"))
        .persist(StorageLevel.MEMORY_AND_DISK)
    )
    labels_df.count()

    changed = True
    iteration = 0

    # cached node count once; useful for deciding whether broadcast is reasonable
    node_count = labels_df.count()

    while changed and iteration < max_iter:
        iteration += 1

        # Broadcast labels only when small enough
        labels_for_join = F.broadcast(labels_df) if node_count <= broadcast_threshold else labels_df

        propagated = (
            df_graph
            .join(labels_for_join, df_graph["dst"] == labels_for_join["node"])
            .select(
                df_graph["src"].alias("node"),
                labels_for_join["label"].alias("neighbor_label")
            )
        )

        min_neighbor_labels = (
            propagated
            .groupBy("node")
            .agg(F.min("neighbor_label").alias("min_neighbor_label"))
            .repartition(num_partitions, "node")
        )

        next_labels_df = (
            labels_df
            .join(min_neighbor_labels, on="node", how="left")
            .select(
                F.col("node"),
                F.least(
                    F.col("label"),
                    F.coalesce(F.col("min_neighbor_label"), F.col("label"))
                ).alias("label")
            )
            .repartition(num_partitions, "node")
            .persist(StorageLevel.MEMORY_AND_DISK)
        )

        # Materialize once
        next_labels_df.count()

        # Checkpoint only every few iterations to cut lineage
        if iteration % checkpoint_every == 0:
            next_labels_df = next_labels_df.localCheckpoint(eager=True).persist(StorageLevel.MEMORY_AND_DISK)
            next_labels_df.count()

        old_df = labels_df.alias("old")
        new_df = next_labels_df.alias("new")

        num_changes = (
            old_df
            .join(new_df, on="node")
            .filter(F.col("old.label") != F.col("new.label"))
            .count()
        )

        if verbose:
            print(f"Iteration {iteration} -> changed labels: {num_changes}")

        changed = num_changes > 0

        labels_df.unpersist()
        labels_df = next_labels_df

    return labels_df, iteration

In [ ]:
final_labels_df, df_iterations = run_ccf_dataframe_fast(df_undirected, max_iter=20, verbose=True)

print("DataFrame iterations =", df_iterations)

(
    final_labels_df
    .select(F.col("node"), F.col("label").alias("component"))
    .orderBy("node")
    .show(truncate=False)
)

In [ ]:
def generate_multi_component_graph(num_components: int, component_size: int) -> DataFrame:
    edges = []
    for c in range(num_components):
        start = c * component_size + 1
        end = start + component_size - 1
        for i in range(start, end):
            edges.append((str(i), str(i + 1)))
    return spark.createDataFrame(edges, ["src", "dst"])

In [ ]:
def benchmark_rdd(df_undirected: DataFrame):
    start = time.time()
    _, iterations = run_ccf_rdd_fast(df_undirected, max_iter=100, verbose=False)
    duration_ms = int((time.time() - start) * 1000)
    return duration_ms, iterations

def benchmark_dataframe(df_undirected: DataFrame):
    start = time.time()
    _, iterations = run_ccf_dataframe_fast(df_undirected, max_iter=100, verbose=False)
    duration_ms = int((time.time() - start) * 1000)
    return duration_ms, iterations

In [ ]:
test_configs = [
    (5, 10),     # 50 nodes
    (5, 20),   # 100 nodes
    (10, 20),   # 200 nodes
    (10, 40)    # 400 nodes
]

In [ ]:
benchmark_results = []

for num_components, component_size in test_configs:
    print(f"Running config: components={num_components}, size={component_size}")

    df_graph = generate_multi_component_graph(num_components, component_size)
    df_undirected_bench = make_undirected(df_graph).cache()
    df_undirected_bench.count()

    num_nodes = num_components * component_size
    num_edges = df_graph.count()

    rdd_time_ms, rdd_iterations = benchmark_rdd(df_undirected_bench)
    df_time_ms, df_iterations = benchmark_dataframe(df_undirected_bench)

    benchmark_results.append((
        num_nodes,
        num_edges,
        rdd_time_ms,
        rdd_iterations,
        df_time_ms,
        df_iterations
    ))

    df_undirected_bench.unpersist()

benchmark_results

In [ ]:
results_df = spark.createDataFrame(
    benchmark_results,
    ["num_nodes", "num_edges", "rdd_time_ms", "rdd_iterations", "df_time_ms", "df_iterations"]
)

results_df.show(truncate=False)